In [4]:
import os

import pandas as pd

In [5]:
# Get the absolute path of the script
current_dir = os.getcwd()

# Move up two levels to the project root
project_root = os.path.abspath(os.path.join(current_dir, "../.."))

# Construct the CSV path
csv_path = os.path.join(project_root, "data", "reviews_labeled_cleaned.csv")

In [6]:
df = pd.read_csv(os.path.join(project_root, "data", "reviews_labeled_cleaned_bench_language.csv"))

In [7]:
lang_name_to_code = {
    "English": "en",
    "French": "fr",
    "Spanish": "es",
    "German": "de",
    "Italian": "it",
    "Dutch": "nl",
    "Portuguese": "pt",
    "Russian": "ru",
    "Japanese": "ja",
    "Chinese": "zh",
    "Korean": "ko",
    "Arabic": "ar",
    "Turkish": "tr",
    "Derja": "dz",
}

def lang_name_to_code_func(lang_name):
    languages = lang_name.split(",")
    languages = [lang.strip() for lang in languages]
    return ",".join([lang_name_to_code[lang_name] for lang_name in languages])


def land_code_standardize(code):
    if code == "zh-cn":
        return "zh"
    if "-latn" in code:
        return code.replace("-latn","")
    return code

import re
import unicodedata


def clean_text(text):
    # Normalize Unicode characters (fix accented letters, etc.)
    text = unicodedata.normalize("NFKC", text)

    # Remove emojis and special symbols
    text = re.sub(r"[^\w\s,.!?]", "", text)  # Keeps punctuation like ,.!?

    # Remove excessive spaces and new lines
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [8]:
df["Language"] = df["Language"].apply(lang_name_to_code_func)
language_columns = ["lang", "lang_langid", "lang_polyglot", "lang_cld3", "lang_fasttext"]
for column in language_columns:
    df[column] = df[column].apply(land_code_standardize)

In [9]:
def score(column1, column2):
    lables = column2.split(",")
    if len(lables) == 1:
        if column1 == column2:
            return 1
        return 0
    if column1 in lables:
        return 0.5
    return 0

In [10]:
for column in language_columns:
    df[column + "_score"] = df.apply(lambda x: score(x[column], x["Language"]), axis=1)

In [11]:
df.groupby("Language").agg({"Language":"count","lang_score": "sum", "lang_langid_score": "sum", "lang_polyglot_score": "sum", "lang_cld3_score": "sum", "lang_fasttext_score": "sum"})

,Language,lang_score,lang_langid_score,lang_polyglot_score,lang_cld3_score,lang_fasttext_score
Language,,,,,,
ar,230,224.0,212.0,223.0,210.0,225.0
"ar,dz",12,6.0,6.0,6.0,6.0,6.0
"ar,dz,fr",1,0.5,0.5,0.5,0.5,0.5
"ar,en",3,1.5,1.5,1.5,1.0,1.5
"ar,fr",2,1.0,1.0,1.0,1.0,1.0
dz,41,0.0,0.0,0.0,0.0,0.0
"dz,ar",6,3.0,3.0,3.0,3.0,3.0
"dz,fr",4,1.5,1.5,1.5,1.5,1.5
en,236,204.0,213.0,232.0,193.0,229.0
